# 26 — When did the switchers turn? The CES panel in 2020, 2022 and 2024

**Question.** Notebook 25 shows the switchers' picture of Trump changed wholesale between 2020 and 2024. The CES panel has a 2022 wave in between. Did the switchers' break with the Democratic side — disapproving of Biden, voting Republican for the House, leaving the party label, turning on its health-care proposals — happen by 2022, or only in the 2024 campaign?

**Plan, fixed before results.** CES 2020–24 panel; Biden 2020 voters by their 2020 report (`CC20_410`), grouped by 2024 vote (loyal = Harris, switch = Trump); weight `commonpostweight_24`. Measures by wave: approval of the sitting president (`CCyy_320a`; Trump in 2020, Biden in 2022 and 2024), identifying as a Democrat including leaners (`pid7` ≤ 3), rating the national economy worse (`CCyy_302` 4–5), Medicare for all and keeping the ACA (items as notebook 21), the four-item immigration index, and in 2022 the party of the House vote (`CC22_412` mapped through `HouseCand{k}Party_post_22`). Reported as weighted shares with unweighted counts; bootstrap 95% intervals on the switcher share.

In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220)
hc = [f'HouseCand{k}Party_post_22' for k in range(1, 9)]
cols = ['CC20_410', 'CC24_410', 'commonpostweight_24', 'commonweight_24', 'CC20_320a', 'CC22_320a', 'CC24_320a', 'CC20_302', 'CC22_302', 'CC24_302',
        'pid7_20', 'pid7_22', 'pid7_24', 'CC20_327a', 'CC22_327a', 'RC24_327a', 'CC20_327d', 'CC22_327c', 'RC24_327c',
        'CC20_331a', 'CC20_331b', 'CC20_331d', 'CC20_331e', 'CC22_331a', 'CC22_331b', 'CC22_331c', 'CC22_331d', 'CC24_331a', 'CC24_331b', 'RC24_331c', 'RC24_331d', 'CC22_412'] + hc
w = pd.read_csv('../data/raw/ces_panel/merged_recontact_2024_vv.csv', usecols=cols, low_memory=False)
w['wt'] = w.commonpostweight_24.fillna(w.commonweight_24)
b = w[(w.CC20_410 == 1) & w.CC24_410.isin([1, 2])].copy(); b['grp'] = np.where(b.CC24_410 == 2, 'switch', 'loyal')
yes = lambda s, vals: s.isin(vals).astype(float).where(s.notna())
M = {}
M['approves Trump (2020)'] = {'2020': yes(b.CC20_320a, [1, 2]).where(b.CC20_320a.between(1, 4))}
M['approves Biden'] = {'2022': yes(b.CC22_320a, [1, 2]).where(b.CC22_320a.between(1, 4)), '2024': yes(b.CC24_320a, [1, 2]).where(b.CC24_320a.between(1, 4))}
M['identifies as Democrat (incl. leaners)'] = {y: yes(b[f'pid7_{y[2:]}'], [1, 2, 3]).where(b[f'pid7_{y[2:]}'].between(1, 7)) for y in ['2020', '2022', '2024']}
M['economy got worse'] = {y: yes(b[f'CC{y[2:]}_302'], [4, 5]).where(b[f'CC{y[2:]}_302'].between(1, 5)) for y in ['2020', '2022', '2024']}
M['supports Medicare for all'] = {'2020': yes(b.CC20_327a, [1]).where(b.CC20_327a.isin([1, 2])), '2022': yes(b.CC22_327a, [1]).where(b.CC22_327a.isin([1, 2])), '2024': yes(b.RC24_327a, [1]).where(b.RC24_327a.isin([1, 2]))}
M['opposes repealing the ACA'] = {'2020': yes(b.CC20_327d, [2]).where(b.CC20_327d.isin([1, 2])), '2022': yes(b.CC22_327c, [2]).where(b.CC22_327c.isin([1, 2])), '2024': yes(b.RC24_327c, [2]).where(b.RC24_327c.isin([1, 2]))}
imm = {'2020': ['CC20_331a', 'CC20_331b', 'CC20_331d', 'CC20_331e'], '2022': ['CC22_331a', 'CC22_331b', 'CC22_331c', 'CC22_331d'], '2024': ['CC24_331a', 'CC24_331b', 'RC24_331c', 'RC24_331d']}
def immidx(cs):
    a = [(b[cs[0]] == 2), (b[cs[1]] == 1), (b[cs[2]] == 1), (b[cs[3]] == 1)]
    return pd.concat([x.astype(float).where(b[c].isin([1, 2])) for x, c in zip(a, cs)], axis=1).mean(axis=1)
M['immigration conservatism (0-1)'] = {y: immidx(cs) for y, cs in imm.items()}
party22 = pd.Series(np.nan, index=b.index)
for k in range(1, 9):
    m = b.CC22_412 == k; party22[m] = b.loc[m, f'HouseCand{k}Party_post_22']
b['house22_dem'] = np.where(party22.astype(str).str.startswith('Democrat'), 1.0, np.where(party22.astype(str).str.startswith('Republican'), 0.0, np.nan))
M['voted Democratic for House, 2022 (of voters)'] = {'2022': b.house22_dem}
rng = np.random.default_rng(26); rows = []
for name, waves in M.items():
    for y, s in waves.items():
        r = {'measure': name, 'wave': y}
        for g in ['loyal', 'switch']:
            m = (b.grp == g) & s.notna(); r[g] = np.average(s[m], weights=b.wt[m]); r[f'N_{g}'] = int(m.sum())
        m = (b.grp == 'switch') & s.notna(); ss, ww = s[m].values, b.wt[m].values
        bs = [np.average(ss[i], weights=ww[i]) for i in (rng.integers(0, len(ss), len(ss)) for _ in range(2000))]
        r['switch_lo'], r['switch_hi'] = np.percentile(bs, [2.5, 97.5]); r['switch_count_yes'] = int(s[m].sum()) if set(np.unique(ss)) <= {0.0, 1.0} else np.nan
        rows.append(r)
t = pd.DataFrame(rows); t.to_csv('../outputs/tables/ces_panel_switch_timing.csv', index=False)
print(t.round(2).to_string(index=False))

                                     measure wave  loyal  N_loyal  switch  N_switch  switch_lo  switch_hi  switch_count_yes
                       approves Trump (2020) 2020   0.01     1642    0.05        39       0.01       0.14               6.0
                              approves Biden 2022   0.89     1632    0.31        38       0.09       0.61              14.0
                              approves Biden 2024   0.89     1634    0.24        39       0.03       0.52               7.0
      identifies as Democrat (incl. leaners) 2020   0.87     1631    0.85        37       0.66       0.94              22.0
      identifies as Democrat (incl. leaners) 2022   0.90     1632    0.75        38       0.48       0.89              17.0
      identifies as Democrat (incl. leaners) 2024   0.91     1641    0.56        39       0.25       0.80              13.0
                           economy got worse 2020   0.95     1633    0.89        38       0.67       1.00              33.0
        